# Performance Analysis: Conway's Game of Life

This notebook benchmarks the Game of Life simulation across different grid sizes and processing strategies (single-process vs multiprocessing with varying worker counts). The goal is to determine the crossover point where multiprocessing becomes advantageous over single-process execution, and use this to set the `multiprocessing_threshold_cells` configuration default.

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Add src/ to path so we can import game_of_life
sys.path.append(str(Path().resolve().parent / 'src'))

from game_of_life.config import SimulationConfig, BoundaryMode
from game_of_life.parallel.dispatch import SingleProcessDispatcher, MultiprocessDispatcher

In [ ]:
grid_sizes = [20, 50, 100, 200, 500, 1000]
strategies = {
    "Single Process": {"type": "single"},
    "Multiprocess (2 Workers)": {"type": "multi", "workers": 2},
    "Multiprocess (4 Workers)": {"type": "multi", "workers": 4},
    "Multiprocess (8 Workers)": {"type": "multi", "workers": 8},
}

num_generations = 20
results = []

for size in grid_sizes:
    shape = (size, size)
    print(f"Benchmarking {size}x{size} (Total Cells: {size*size})")
    
    # Generate random initial state
    initial_grid = np.random.choice([0, 1], size=shape, p=[0.8, 0.2]).astype(np.uint8)
    
    for name, strategy_config in strategies.items():
        grid_copy = initial_grid.copy()
        
        config = SimulationConfig(boundary_mode=BoundaryMode.TOROIDAL)
        
        if strategy_config["type"] == "single":
            dispatcher = SingleProcessDispatcher(boundary_mode=BoundaryMode.TOROIDAL)
        else:
            config.n_workers = strategy_config["workers"]
            dispatcher = MultiprocessDispatcher(shape=shape, config=config, initial=grid_copy)
            
        # Warmup
        dispatcher.step(grid_copy)
        
        # Benchmark
        start_ns = time.perf_counter_ns()
        for _ in range(num_generations):
            _, _, _ = dispatcher.step(grid_copy)
        end_ns = time.perf_counter_ns()
        
        dispatcher.shutdown()
        
        mean_time_ms = ((end_ns - start_ns) / 1_000_000.0) / num_generations
        
        results.append({
            "Grid Size": size,
            "Total Cells": size * size,
            "Strategy": name,
            "Mean Time (ms)": mean_time_ms
        })
        print(f"  {name}: {mean_time_ms:.2f} ms / gen")
        
df = pd.DataFrame(results)

In [ ]:
import seaborn as sns
sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 6))
sns.lineplot(data=df, x="Total Cells", y="Mean Time (ms)", hue="Strategy", marker="o")

plt.xscale("log")
plt.yscale("log")
plt.title("Game of Life Performance: Mean Time per Generation vs Grid Size")
plt.xlabel("Total Cells (log scale)")
plt.ylabel("Mean Time per Generation in ms (log scale)")
plt.legend(title="Strategy")
plt.tight_layout()
plt.show()

## Conclusion

Based on the generated plot, we can observe the crossover point where multiprocessing overhead is overcome by parallel speedup. This value should be updated in `config.yaml` / `config.py` as the `multiprocessing_threshold_cells` value, and noted in `context.md`.